## Conclusions from a niave approach:
- Counting the number of opperations comparing string simiarlity of each item with every other results in $\frac{n(n-1)}{2}$ <br>the testing dataset ts consists of 1,000,000 rows and so one would expect to peform ≈ <span style="color:red">500 billion opperations</span> <br>
- **Experements have estimated around Comparisions/sec = 21,214.  One would thus expect to wait for the process to complete for 6547 hours. Not a practical approach.** <br><br>
<br><br>
The severity of this situation can be remedied slightly by deciding a effective blocking stratergy.  <br>
For example:<br>
 
    In the testing set,  The class balance is roughly split evenly between US and India adresses. (59% US and 40% India [other countrys in testing]) <br>
    Splitting 1,000,000 the data down the midde, to subsets of n(n-1/2)  => (500,000*500,000)/2 *2 => <span style="color:red">250 billion opperations.</span>  ie half time.

## Class exploration for GPE 
See csv
```bash
"Training set Countries",
    "           count        pct",
    "country                    ",
    "US       7510506  59.954355",
    "India    5016534  40.045645",
    
    "Testing set Countries",
    "           count        pct",
    "country                    ",
    "India    5527551  47.235414",
    "US       4480137  38.284790",
    "France   1694445  14.479796",
    "Processed 200,000 rows",
    "Top GPEs so far:
    "[('DELHI', 14307), ('NC', 8372), ('MUMBAI', 7489), ('NEW DELHI', 7266), ('KARNATAKA', 6222), ('VA', 5350), ('GUJARAT', 5264), ('TELANGANA', 5003), ('WEST BENGAL', 4876), ('KOLKATA', 4629)]",
    "Processing rows:   2%|▏         | 201483/12527040 [04:41<5:06:34, 670.07it/s]\n",
    ```

In [3]:
import pandas as pd 
import matplotlib.pyplot as plt
from utils.import_data import ImportData


C:\Users\Graem\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [6]:
train = ImportData().import_training_data()
USA_train = train[train['country'] == 'US']

test = ImportData().import_testing_data()
USA_test = test[test['country'] == 'US']

In [38]:
import re

#USA_train["adress_parts"] = USA_train["business_address"].str.split(",")
USA_train["State"] = USA_train["business_address"].str.extract(
    r"(?:^|,\s*)([A-Z]{2})(?=\s*,|$)"
)
USA_train.head(10)


,entity_id,business_name,business_address,country,State
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,NC
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,OK
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,AZ
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,MD
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US,OH
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US,TN
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US,IN
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US,WI
10,S1-504790211,Crystal Lending PC,"11643 Prosperity Road, South Jordan, UT",US,UT
12,S1-236312382,"Kelly Advisory, Inc","301 1st Street, Chokio, MN",US,MN


In [39]:
state_abbreviations = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
}

In [40]:
USA_train[USA_train["State"].notnull()]["State"].unique()

<StringArray>
['NC', 'OK', 'AZ', 'MD', 'OH', 'TN', 'IN', 'WI', 'UT', 'MN', 'CT', 'VA', 'MO',
 'DE', 'WA', 'ME', 'MA', 'WV', 'IL', 'NY', 'TX', 'CA', 'AL', 'PA', 'IA', 'KY',
 'LA', 'NM', 'AR', 'KS', 'OR', 'VT', 'AK', 'FL', 'GA', 'ND', 'NE', 'DC', 'SC',
 'RI', 'CO', 'MT', 'WY', 'ID', 'SD', 'PG', 'NN', 'OU', 'AA', 'KW', 'LS', 'AI',
 'JJ', 'GG', 'UU', 'HR', 'WH', 'TT', 'VV', 'II', 'DW', 'FF', 'HH', 'EE', 'CM',
 'CC', 'CS', 'KK', 'MM', 'XX', 'ZB']
Length: 71, dtype: str

In [47]:
unrecognized_state_abbreviations = sorted(
    observed_states - set(state_abbreviations.values())
)

unrecognized_state_abbreviations


['AA',
 'AI',
 'CC',
 'CM',
 'CS',
 'DC',
 'DW',
 'EE',
 'FF',
 'GG',
 'HH',
 'HR',
 'II',
 'JJ',
 'KK',
 'KW',
 'LS',
 'MM',
 'NN',
 'OU',
 'PG',
 'TT',
 'UU',
 'VV',
 'WH',
 'XX',
 'ZB']

In [51]:
USA_train[USA_train["State"] == "AA"].head(1)

unrecognized_state_counts = (
    USA_train.loc[
        USA_train["State"].isin(unrecognized_state_abbreviations), "State"
    ]
    .value_counts()
    .reindex(unrecognized_state_abbreviations, fill_value=0)
)

print(unrecognized_state_counts)
print("Total:", unrecognized_state_counts.sum())

State
AA        4
AI        1
CC        2
CM        1
CS        1
DC    22422
DW        1
EE        1
FF        2
GG        2
HH        3
HR        1
II        3
JJ        2
KK        1
KW        1
LS        1
MM        1
NN        3
OU        4
PG       77
TT        1
UU        1
VV        3
WH        2
XX        1
ZB        1
Name: count, dtype: int64
Total: 22543
